# LTX Portrait Student 14×2048 — полное обучение на CelebV-HQ

Ученик: **14×2048** (~983M параметров). Блоки `[0, 2, 4, 6, 8, 10, 12, 15, 17, 19, 21, 23, 25, 27]` копируются из 28-слойного учителя.

Подготовьте официальный CelebV-HQ:

```bash
python download_celebv_hq.py --accept-license
```

Pipeline использует весь доступный CelebV-HQ:

- все клипы участвуют в train, а 1% regular-compatible клипов дополнительно образует стабильный validation-набор;
- все action-категории, включая talking, singing, eating, laughing и активную мимику, дают real-video flow/x0 supervision;
- технически непригодные клипы всегда идут в image-only teacher rollout, а compatible-клипы распределяются между real-video и rollout по curriculum;
- conditioning всегда берётся из CelebV-HQ, отдельный датасет изображений не нужен;
- 23.976/24 FPS обрабатываются без повторённых кадров;
- rollout curriculum плавно растёт по шагам с 40% до 50%;
- 65% rollout используют captions CelebV-HQ, 35% — motion/camera prompt bank с комбинированными wind+camera сценариями;
- rollout-окна равномерно покрывают доступную траекторию, CFG равен 3.0;
- последние три rollout-шагa дифференцируемы, teacher корректирует текущее состояние student по x0 и temporal dynamics;
- feature distillation работает до конца обучения, real-video flow/x0 сохраняют значимый вес;
- checkpoint выбирается по teacher-relative endpoint, temporal, motion и decoded LPIPS;
- пользовательские preview prompts находятся отдельно в `preview_prompts.py` и не меняют обучение.

Новые веса пишутся в `outputs_ltx_student_celebv_hq`. Перед стартом удалите старые checkpoints, если архитектура изменилась.


In [ ]:
from dataclasses import asdict
from pathlib import Path
import importlib

import celebv_hq_data
import ltx_portrait_student

# Удобно после правок модулей: не требуется перезапуск kernel.
importlib.reload(celebv_hq_data)
importlib.reload(ltx_portrait_student)
from ltx_portrait_student import PortraitStudentConfig, PortraitStudentTrainer

cfg = PortraitStudentConfig(
    dataset_root="celebv_hq",
    manifest_name="celebv_hq_manifest.csv",
    output_dir="outputs_ltx_student_celebv_hq",
    frames=49,
    fps=25,
    train_on_full_dataset=True,
    student_num_layers=14,
    student_num_attention_heads=32,
    student_attention_head_dim=64,
    epochs=2,
    validation_fraction=0.01,
    validation_eval_fraction=1.0,
    rollout_probability=0.40,
    final_rollout_probability=0.50,
    rollout_prompt_bank_probability=0.35,
    rollout_guidance_scale=3.0,
    rollout_backprop_steps=3,
    rollout_validation_batches=64,
    endpoint_validation_batches=8,
    ema_decay=0.9995,
)

print(asdict(cfg))

In [ ]:
# Загружает VAE, T5 и teacher transformer, затем строит и инициализирует студента.
trainer = PortraitStudentTrainer(cfg)

print("student config:", dict(trainer.student.config))
print("total optimizer steps:", trainer.total_optimizer_steps)

In [ ]:
# Обязательный smoke test до долгого запуска: VAE encode, mask, RoPE и оба transformer forward.
# Здесь веса не изменяются.
sanity_metrics = trainer.sanity_check()
sanity_metrics

In [ ]:
# Долгий запуск. Resume происходит автоматически из checkpoint-latest.
# Возвращается checkpoint-best по teacher-relative rollout/endpoint + LPIPS;
# EMA-кандидат лежит рядом в checkpoint-best/ema_transformer.
best_path = trainer.train()
print("selected checkpoint:", best_path)

In [ ]:
# Последние численные метрики. Для визуальной оценки после checkpoint-best
# запустите generate_ltx_student_preview.ipynb.
history_path = Path(cfg.output_dir) / "history.jsonl"
if history_path.exists():
    history_lines = history_path.read_text(encoding="utf-8").splitlines()
    print("history records:", len(history_lines))
    print(*history_lines[-5:], sep="\n")
else:
    print("history пока не создан")

In [ ]:
# Освободить GPU после обучения/валидации.
trainer.close()